Optimisation paramétrique (threshold et bin_size)

In [ ]:
import fmatoolbox as fma
from math import *
import matplotlib.pyplot as plt
import numpy as np
import regions as rg
from scipy import signal
from scipy.stats import spearmanr
%load_ext autoreload
%autoreload 2

In [ ]:
session = '/mnt/hubel-data-131/perceval/Rat003_20231219/Rat003_20231219.xml'

In [ ]:
R = rg.data.Regions(session,states=['sws','rem'],events=['ripples'])
spikes = R.spikes(regs=["nr"], state='sws')
fr = R.firingRate(regs=["nr"], states=['sws'], window=0.01, step=1)

Fonction de score

In [ ]:
def avalanchesresult(session, spikes=None, spikes_indiv=None, FR=None, bin_size=0.05, neurons=None, tmax=50, threshold=30, window=5):#neurons = nombre de neurones à prendre en compte, tmax = durée max à prendre en compte, threshold = seuil pour définir les avalanches, smoothed = si on veut afficher la courbe lissée, window = taille de la fenêtre pour le lissage 
    
    if spikes_indiv==None:
        spikes_indiv = fma.data.loadSpikeTimes(session)
    if neurons is not None:
        spikes_indiv = dict(list(spikes_indiv.items())[:neurons])
    spikes_indiv = {key: fma.general.restrict(spikes_indiv[key], [0, tmax]) for key in spikes_indiv} 
    

    if spikes.all()==None:
        spikes = fma.data.loadSpikeTimes(session, output='compact')
    if neurons is not None:
        ind = spikes[:,1] < neurons
        spikes = spikes[ind,:]
    spikes = spikes[(spikes[:,0] >= 0) & (spikes[:,0] <= tmax),:]
    if FR is None:
        FR = fma.analysis.firingRate(spikes[:,0], bin_size=bin_size)
    FR = FR[FR[:,0] <= tmax,:]
    time = FR[:, 0]
    rates = FR[:, 1]
    size, intervals, size_t = rg.computation.avalanchesFromProfile(
        rates,
        threshold=threshold,
        time_step=FR[1,0] - FR[0,0],
        t0=FR[0,0]
    )

    return spikes, FR, size, intervals, size_t

In [ ]:
def avalanche_score(session, bin_size, threshold, spikes, spikes_indiv, tmax=10000, w1=0.8, w2=0.0, w3=0.0):
    spikes, FR, size, intervals, size_t = avalanchesresultsession, spikes, spikes_indiv, tmax=tmax, bin_size=bin_size, threshold=threshold)
    if len(size) < 2:
        return -np.inf

    size = np.array(size)

    n_aval = len(size)
    var_size = np.var(size)

    total_time = FR.shape[0] * bin_size
    active_time = sum([t2-t1 for t1,t2 in intervals])
    frac_active = active_time / total_time

    p1 = (var_size / np.mean(size)**2)
    p3 = (n_aval / (total_time/np.mean([t2-t1 for t1,t2 in intervals])))

    # pénalités
    score = (
    w1 * p1
    - w2 * frac_active
    - w3 * p3
    )

    print(f"Score: {score:.2f} (var_size={p1:.2f}, frac_active={frac_active:.2f}, n_aval={p3})")

    return score


In [ ]:
R = rg.data.Regions(session,states=['sws','rem'],events=['ripples'])
spikes = R.spikes(regs=["nr"], state='sws')

spikes_indiv = {}
d = {}
for value, neuron in spikes:
    neuron = np.uint(neuron)
    if neuron not in d:
        d[neuron] = []
    d[neuron].append(value)
for neuron, values in d.items():
    spikes_indiv[neuron] = values

Optimisation du bin_size pour threshold = 0

In [ ]:
bin_range = np.linspace(0.001, 1, 200)

scores = np.zeros(len(bin_range))

for i, b in enumerate(bin_range):
        scores[i] = avalanche_score(session, bin_size=b, threshold=0, spikes=spikes, spikes_indiv=spikes_indiv, tmax=2000)

In [ ]:
fig = plt.figure(figsize=(8,6))

plt.plot(bin_range,scores)

plt.show()

Plot de la loss 3d

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt

# grille 2D
B, T = np.meshgrid(bin_range, th_range, indexing='ij')

# plot 3D
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(B, T, scores, cmap='viridis')

ax.set_xlabel("bin_size")
ax.set_ylabel("threshold")
ax.set_zlabel("score")

fig.colorbar(surf, shrink=0.5, aspect=5)

plt.show()

Heatmap + ridge

In [ ]:
%matplotlib widget

best_bin_indices = np.argmax(scores, axis=0)
best_bins = bin_range[best_bin_indices]

fig, ax = plt.subplots(figsize=(8,6))

im = ax.imshow(
    scores,
    origin='lower',
    aspect='auto',
    extent=[th_range[0], th_range[-1], bin_range[0], bin_range[-1]],
    cmap='viridis'
)

ax.plot(th_range, best_bins, color='red', lw=3, label="optimal bin_size")

ax.set_xlabel("threshold")
ax.set_ylabel("bin_size")
ax.set_title("Score landscape + optimal ridge")

ax.legend()

plt.colorbar(im, label="score")

plt.show()

i_max, j_max = np.unravel_index(np.argmax(scores), scores.shape)

best_bin = bin_range[i_max]
best_th = th_range[j_max]

ax.scatter(best_th, best_bin, color='white', s=120, edgecolor='black', label='global optimum')

Grid search

In [ ]:
bin_range = np.linspace(0.05, 0.5, 4)
th_range = np.linspace(5, 30, 6)

best_score = -np.inf
best_params = None

for b in bin_range:
    for th in th_range:
        s = avalanche_score(session, bin_size=b, threshold=th, spikes=spikes, spikes_indiv=spikes_indiv, tmax=2000)
        if s > best_score:
            best_score = s
            best_params = (b, th)
        print(b, th)

print("Best params:", best_params)

Gradient Descent uniquement sur bin_size

In [ ]:
def optimize_bin_size(session, threshold, b0=0.2, lr=0.01, eps=1e-3, steps=20):

    b = b0

    for i in range(steps):

        s_plus = avalanche_score(session, b + eps, threshold, spikes=spikes, spikes_indiv=spikes_indiv, tmax=2000)
        s_minus = avalanche_score(session, b - eps, threshold, spikes=spikes, spikes_indiv=spikes_indiv, tmax=2000)
        print(s_plus, s_minus)

        grad = (s_plus - s_minus) / (2 * eps)

        b = b + lr * grad

        print(f"step {i}: bin_size={b:.4f}, grad={grad:.4f}")

    return b
#Plutôt qu'une boucle, peut-être plus pertinent d'utiliser un while
#Avec cette fonction de score, le grad peut diverger si eps trop petit


best_bin = optimize_bin_size(session, threshold=30, eps=0.01, lr=0.005, steps=5)
print("Best bin_size:", best_bin)

Bayesian optimization

In [ ]:
import optuna

def objective(trial):

    b = trial.suggest_float("bin_size", 0.05, 0.5)
    th = trial.suggest_float("threshold", 5, 30)

    return avalanche_score(session, b, th, spikes=spikes, spikes_indiv=spikes_indiv, tmax=2000)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(study.best_params)

Rq : Gradient Descent & Bayesian optimization mènent à un bin_size, similaire, pour cette session ~0.375, pour threshold=30

In [ ]:
avalanchesplot(session, tmax=15, smoothed=True, bin_size = 0.375, threshold=30)

La Deviation from Criticality Coefficient est :
DCC=∣γmeasured−γpred∣
-> Matlab ?
Sinon en Python :

In [ ]:
import numpy as np
import powerlaw
from scipy.stats import linregress


def compute_dcc(sizes, durations):

    sizes = np.array(sizes)
    durations = np.array(durations)

    # --- exponent tau (size distribution)
    fit_s = powerlaw.Fit(sizes, discrete=True, verbose=False)
    tau = fit_s.power_law.alpha

    # --- exponent alpha (duration distribution)
    fit_t = powerlaw.Fit(durations, discrete=True, verbose=False)
    alpha = fit_t.power_law.alpha

    # --- gamma measured
    unique_T = np.unique(durations)

    mean_sizes = []
    for T in unique_T:
        mean_sizes.append(np.mean(sizes[durations == T]))

    mean_sizes = np.array(mean_sizes)

    slope, _, _, _, _ = linregress(
        np.log(unique_T),
        np.log(mean_sizes)
    )

    gamma_measured = slope

    # --- predicted gamma
    gamma_pred = (alpha - 1) / (tau - 1)

    # --- DCC
    dcc = abs(gamma_measured - gamma_pred)

    return dcc, tau, alpha, gamma_measured

In [ ]:
def avalanche_loss(session, bin_size, threshold):

    spikes, FR, sizes, intervals, size_t = avalanchesresult(session, neurons=15, tmax=15, smoothed=False, bin_size=bin_size, threshold=threshold)
    durations = np.array(intervals[:,1])-np.array(intervals[:,0])

    if len(sizes) < 2:
        return np.inf, np.inf

    dcc, tau, alpha, gamma = compute_dcc(sizes, durations)
    print(tau)

    return dcc, gamma

Pareto front

In [ ]:
import optuna
import numpy as np

def make_objective(session, spikes, spikes_indiv, tmax):
    def objective(trial):
        b = trial.suggest_float("bin_size", 0.05, 0.5)
        th = trial.suggest_float("threshold", 5, 30)
        _, FR, size, intervals, size_t = avalanchesresult(session, spikes, spikes_indiv, tmax=tmax, bin_size=b, threshold=th)
        if len(size) < 2:
            return -np.inf

        size = np.array(size)

        n_aval = len(size)
        var_size = np.var(size)

        total_time = FR.shape[0] * b
        active_time = sum([t2-t1 for t1,t2 in intervals])
        frac_active = active_time / total_time

        p1 = (var_size / np.mean(size)**2)
        p3 = (n_aval / (total_time/np.mean([t2-t1 for t1,t2 in intervals])))


    # Optionnel : pénalité si trop extrême
    # n_target = 200
    # penalty = -abs(n - n_target)

        return p1,p3
    return objective

In [ ]:
R = rg.data.Regions(session,states=['sws','rem'],events=['ripples'])
spikes = R.spikes(regs=["nr"], state='sws')

spikes_indiv = {}
d = {}
for value, neuron in spikes:
    neuron = np.uint(neuron)
    if neuron not in d:
        d[neuron] = []
    d[neuron].append(value)
for neuron, values in d.items():
    spikes_indiv[neuron] = values

objective = make_objective(session, spikes, spikes_indiv, 2000)
study = optuna.create_study(
    directions=["maximize", "minimize"]  # variance ↑, nombre ↑
)

study.optimize(objective, n_trials=100)

In [ ]:
pareto_trials = study.best_trials

for t in pareto_trials:
    print("Params:", t.params)
    print("Variance:", t.values[0], " | N avalanches:", t.values[1])

In [ ]:
import matplotlib.pyplot as plt

# Récupérer toutes les valeurs
trials = study.trials

variance = [t.values[0] for t in trials if t.values is not None]
n_avalanches = [t.values[1] for t in trials if t.values is not None]

# Pareto front
pareto_trials = study.best_trials
pareto_var = [t.values[0] for t in pareto_trials]
pareto_n = [t.values[1] for t in pareto_trials]

# Plot
plt.figure()
plt.scatter(n_avalanches, variance, label="All trials")
plt.scatter(pareto_n, pareto_var, label="Pareto front")

plt.xlabel("Number of avalanches")
plt.ylabel("Variance")
plt.title("Pareto Front - Avalanche Optimization")
plt.legend()

plt.show()